# Package

In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [79]:
import importlib

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} is not installed. Installing via !pip ...")
        try:
            get_ipython().system(f"pip install {package_name}")
            print(f"{package_name} installed successfully.")
        except Exception as e:
            print(f"Failed to install {package_name}. Error:\n{e}")

packages = ["pyDOE2", "diversipy", "pygmo", "optproblems", "pymoo"]

for pkg in packages:
    ensure_package(pkg)

pyDOE2 is already installed.
diversipy is already installed.
pygmo is not installed. Installing via !pip ...
  Using cached pygmo-v2.19.0.tar.gz (3.0 MB)
ERROR: pygmo from https://files.pythonhosted.org/packages/e2/12/090ba61479f60d5177a0048736d09dc028b2d65063ed44cb952df506336f/pygmo-v2.19.0.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
pygmo installed successfully.
optproblems is already installed.
pymoo is already installed.


In [80]:
from pyDOE2 import lhs
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import time
import sys
import os
sys.path.append(os.path.abspath('/content/drive/MyDrive/PhD 2025-1 Offline data-driven MOP under uncertainty/ Prob-RVEA and Prob-MOEA D 2022'))
from pymoo.problems import get_problem
from pymoo.operators.sampling.lhs import LHS
from pymoo.problems.multi.omnitest import OmniTest

from desdeo_problem.Problem import DataProblem
from desdeo_problem.surrogatemodels.SurrogateKriging import SurrogateKriging
from desdeo_problem.testproblems.TestProblems import test_problem_builder

from desdeo_emo.EAs.ProbRVEA import RVEA
from desdeo_emo.EAs.ProbRVEA import ProbRVEA
from desdeo_emo.EAs.ProbRVEA import ProbRVEA_v3
from desdeo_emo.EAs.ProbRVEA import HybRVEA
from desdeo_emo.EAs.ProbRVEA import HybRVEA_v3

from desdeo_emo.EAs.ProbMOEAD import MOEA_D
from desdeo_emo.EAs.ProbMOEAD import ProbMOEAD
from desdeo_emo.EAs.ProbMOEAD import ProbMOEAD_v3
from desdeo_emo.EAs.ProbMOEAD import HybMOEAD
from desdeo_emo.EAs.ProbMOEAD import HybMOEAD_v3

In [81]:
# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Problem

In [82]:
def mean_std(arr):
    return np.mean(arr), np.std(arr)

# Initial settings
# Problem: DTLZ1-7, omnitest

problem_name = 'omnitest'
problem_testbench = 'dtlz'

if 'dtlz' in problem_name:
  nvars = 10
  nobjs = 2
  problem_pymoo = get_problem(problem_name, n_var=nvars, n_obj=nobjs)

elif 'omnitest' in problem_name:
  nvars = 2
  nobjs = 2
  problem_pymoo = OmniTest(n_var=nvars)

#nsamples = 11*nvars-1
nsamples = 1000

print(f"Problem name: {problem_name}")
print(f"Cons: {problem_pymoo.n_constr}")
print(f"Var: {nvars}")
print(f"Obj: {nobjs}")

# Metrics: HV
# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

Problem name: omnitest
Cons: 0
Var: 2
Obj: 2

Min-Max normalization -> Min:  [-2 -2]
Min-Max normalization -> Max:  [2.4 2.4]
HV Reference points:  [1.1 1.1]


In [83]:
# Read dataset
np.random.seed(42)
sampling_pymoo = LHS()
x = sampling_pymoo(problem_pymoo, nsamples, seed=42).get("X")
y = problem_pymoo.evaluate(x, return_values_of=["F"])

print('problem_name', problem_name)
print('x', x.shape)
print('y', y.shape)

problem_name omnitest
x (1000, 2)
y (1000, 2)


In [84]:
x

array([[1.47253591, 3.60173333],
       [5.07193713, 5.97454806],
       [1.53576486, 3.19300391],
       ...,
       [0.05110892, 2.51653149],
       [5.74226104, 5.96195231],
       [0.8895074 , 3.8673152 ]])

In [85]:
# Create problem object
is_data = True
x_names = [f'x{i}' for i in range(1,nvars+1)]
y_names = [f'f{i}' for i in range(1,nobjs+1)]
row_names = ['lower_bound','upper_bound']
if is_data is False:
    prob = test_problem_builder(problem_name, nvars, nobjs)
    x = lhs(nvars, nsamples)
    y = prob.evaluate(x)
    data = pd.DataFrame(np.hstack((x,y.objectives)), columns=x_names+y_names)
else:
    data = pd.DataFrame(np.hstack((x,y)), columns=x_names+y_names)

if problem_testbench == 'DDMOPP':
    x_low = np.ones(nvars)*-1
    x_high = np.ones(nvars)
else:
    x_low = problem_pymoo.xl
    x_high = problem_pymoo.xu
    print('x_low: ', x_low)
    print('x_high: ', x_high)

bounds = pd.DataFrame(np.vstack((x_low,x_high)), columns=x_names, index=row_names)
problem = DataProblem(data=data, variable_names=x_names, objective_names=y_names,bounds=bounds)
start = time.time()
problem.train(SurrogateKriging)
end = time.time()
time_taken = end - start
print("Kriging surrogates built in:",time_taken,"(secs)")

x_low:  [0. 0.]
x_high:  [6. 6.]
Kriging surrogates built in: 3.2876675128936768 (secs)


# Probabilstic MOEA/D

In [86]:
mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, nvars), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, :nobjs-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem_pymoo.evaluate(X_opt)
else:
    pf = problem_pymoo.pareto_front()
igd_plus = IGDPlus(pf)

for seed in range(1,6):

    start_time = time.time()
    evolver_opt = ProbMOEAD_v3(problem,
                               use_surrogates=True,
                               population_size = 50,
                               total_function_evaluations=10000)

    while evolver_opt.continue_evolution():
        evolver_opt.iterate()
    end_time = time.time()

    # Results
    solution = evolver_opt.population.individuals
    obj = evolver_opt.population.objectives
    f_real = problem_pymoo.evaluate(solution, return_values_of=["F"])
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    print(f"\nSeed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"IGD+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f}")


Seed 1 | Time: 635.11s | MSE: 3.81e-05 | IGD+: 4.27e-01 | Sur HV: 1.01 | Real HV: 1.02

Seed 2 | Time: 634.70s | MSE: 4.69e-05 | IGD+: 7.51e-01 | Sur HV: 0.98 | Real HV: 0.98

Seed 3 | Time: 635.46s | MSE: 2.25e-05 | IGD+: 1.85e-01 | Sur HV: 1.10 | Real HV: 1.10

Seed 4 | Time: 639.06s | MSE: 1.09e-05 | IGD+: 7.48e-01 | Sur HV: 0.95 | Real HV: 0.95

Seed 5 | Time: 633.61s | MSE: 2.27e-05 | IGD+: 2.09e-01 | Sur HV: 1.06 | Real HV: 1.06


In [87]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== Prob-MOEA/D: Statistics over 30 runs ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

Problem name:  omnitest

=== Prob-MOEA/D: Statistics over 30 runs ===
MSE: Mean = 2.82e-05, Std = 1.27e-05
IGD+: Mean = 4.64e-01, Std = 2.48e-01
Sur HV: Mean = 1.02, Std = 0.05
Real HV: Mean = 1.02, Std = 0.05


In [88]:
print(problem_name+'_MSE_Prob-MOEA/D'+' =',mse_list)
print(problem_name+'_IGD+_Prob-MOEA/D'+' =',igd_list)
print(problem_name+'_HV_Prob-MOEA/D'+' =',hv_real_list)

omnitest_MSE_Prob-MOEA/D = [3.808008219446484e-05, 4.692412285153483e-05, 2.2453742801760547e-05, 1.0946624117116407e-05, 2.270824639456749e-05]
omnitest_IGD+_Prob-MOEA/D = [0.42652652737810404, 0.7507029127263416, 0.18495691635151287, 0.7478741821837002, 0.20927221795405276]
omnitest_HV_Prob-MOEA/D = [1.0184820383539508, 0.975754172302029, 1.0967110174087014, 0.9506341204613185, 1.0614039707975016]
